# 3.1 - Teknik Evaluasi Model & Agen Generative AI (LLM Evaluation)

Notebook ini memandu pengujian dan evaluasi kualitas respons sistem **RAG (Retrieval-Augmented Generation)** dan **AI Agents** menggunakan pustaka **MLflow GenAI Evaluation** serta metodologi **LLM-as-a-Judge**.

### Ringkasan Proses:
1. **Instalasi Pustaka & Setup**: Menyiapkan modul evaluasi MLflow dan menginisialisasi lingkungan lab.
2. **Memuat Model Agen dari Unity Catalog**: Memuat artefak model terdaftar `dphone_agent` (versi 1) dari katalog Unity Catalog.
3. **Mendefinisikan Predict Wrapper**: Membungkus antarmuka inferensi model agar kompatibel dengan runner evaluasi MLflow.
4. **Evaluasi Tanpa Ground Truth**: Mengukur kualitas respons menggunakan metrik *RelevanceToQuery*, *RetrievalRelevance*, *RetrievalGroundedness*, dan *Safety*.
5. **Evaluasi dengan Ground Truth**: Menilai ketepatan faktual (*Correctness*) dan kecukupan retrieval (*RetrievalSufficiency*) terhadap fakta acuan (*expected facts*).
6. **Evaluasi Berbasis Pedoman Kustom (*Custom Guidelines*)**: Menilai kepatuhan terhadap aturan bisnis (bahasa, nada bicara, serta kebijakan harga & privasi).
7. **Perbandingan Model Penantang (*Challenger Model*)**: Mengevaluasi model alternatif (`@challenger`) untuk membandingkan kinerja model secara objektif (*head-to-head*).


### Langkah 1: Instalasi Library Pendukung
Menginstal paket `databricks-agents`, `mlflow`, `databricks-openai`, dan `uv` untuk fungsionalitas evaluasi GenAI, lalu merestart kernel Python agar dependensi langsung aktif.


In [0]:
# Menginstal dependensi untuk evaluasi GenAI: backoff, databricks-openai, uv, databricks-agents, dan mlflow
%pip install --quiet --upgrade backoff databricks-openai uv databricks-agents mlflow-skinny[databricks]
# Merestart kernel Python agar pustaka yang baru diinstal dapat langsung digunakan
dbutils.library.restartPython()

### Langkah 2: Setup Lingkungan Lab
Menjalankan notebook setup lab untuk menginisialisasi variabel katalog, skema, dan konfigurasi dari `../Includes/Lab_Setup`.


In [0]:
%run ../Includes/Lab_Setup

### Langkah 3: Memuat Model Agen dari Unity Catalog
Memuat model agen terdaftar `dphone_agent` (versi 1) yang telah dideploy pada modul sebelumnya dari Unity Catalog Model Registry.


In [0]:
import mlflow

# Menentukan nama model dan versi yang terdaftar di Unity Catalog
model_name = f"{course.catalog}.{course.schema}.dphone_agent"
model_version = 1
logged_model_uri = f"models:/{model_name}/{model_version}"

# Memuat model pyfunc dari Unity Catalog Model Registry
loaded_model = mlflow.pyfunc.load_model(logged_model_uri)

### Langkah 4: Mendefinisikan Fungsi Pembungkus Prediksi (Predict Wrapper)
Membuat fungsi `predict_wrapper` untuk mengonversi kueri string sederhana menjadi format input JSON yang diharapkan oleh `loaded_model`, serta mengekstrak respons teks akhir dari struktur keluaran agen.


In [0]:
def predict_wrapper(query):
    """Fungsi pembungkus inferensi untuk menerima string kueri dan mengembalikan string jawaban agen."""
    model_input = {
        "input": [
            {"role": "user", "content": query}
        ]
    }
    # Menjalankan inferensi model dengan format input yang sesuai
    response = loaded_model.predict(model_input)

    # Mengekstrak teks jawaban terakhir dari struktur keluaran respon agen
    return response["output"][-1]["content"][0]["text"]

### Langkah 5: Pengujian Fungsi Wrapper secara Mandiri
Menguji eksekusi `predict_wrapper` dengan pertanyaan seputar kapasitas penyimpanan produk untuk memastikan model merespons dengan benar.


In [0]:
# Menguji eksekusi fungsi wrapper dengan pertanyaan sampel tentang kapasitas penyimpanan
predict_wrapper("What is the storage capacity of dPhone D1 Pro?")

### Langkah 6: Menyiapkan Dataset Evaluasi Tanpa Ground Truth
Membuat dataset evaluasi berisi daftar pertanyaan pengguna terbuka (*unlabeled dataset*) tanpa menyertakan jawaban ekspektasi (`"expectations": None`).


In [0]:
# Dataset evaluasi berisi pertanyaan terbuka tanpa jawaban patokan (unlabeled dataset)
eval_dataset = [
    {
        "inputs": {'query': 'How to replace the battery of dPhone D1?'},
        "expectations": None
    },
    {
        "inputs": {'query': 'Explain how the dPhone D1 Pro is waterproof?'},
        "expectations": None
    }
]

### Langkah 7: Evaluasi LLM-as-a-Judge Tanpa Ground Truth
Menjalankan evaluasi otomatis menggunakan model LLM sebagai juri (*LLM Judge*):
- `RelevanceToQuery`: Menilai apakah jawaban relevan dengan pertanyaan yang diajukan.
- `RetrievalRelevance`: Menilai apakah dokumen yang di-retrieve relevan dengan topik pertanyaan.
- `RetrievalGroundedness`: Menilai apakah jawaban didukung penuh oleh dokumen acuan (mendeteksi halusinasi).
- `Safety`: Memastikan jawaban bebas dari konten berbahaya, toksik, atau pelanggaran keamanan.


![llm-judges.png](../Includes/Images/llm-judges.png "LLM Judges")

*Diagram di atas mengilustrasikan alur evaluasi LLM-as-a-Judge tanpa ground truth: model juri menilai relevansi kueri, relevansi dokumen yang di-retrieve, keterlandasan jawaban, dan keamanan.*


In [0]:
from mlflow.genai.scorers import RelevanceToQuery, RetrievalRelevance, RetrievalGroundedness, Safety

# Menjalankan evaluasi GenAI otomatis menggunakan LLM-as-a-Judge tanpa fakta acuan
eval_results = mlflow.genai.evaluate(
    predict_fn=predict_wrapper,
    data=eval_dataset,
    scorers=[RelevanceToQuery(), RetrievalRelevance(), RetrievalGroundedness(), Safety()],
)

### Langkah 8: Menyiapkan Dataset Evaluasi dengan Fakta Acuan (Ground Truth)
Menyusun dataset pengujian yang dilengkapi dengan fakta acuan (`expected_facts`) atau teks jawaban yang diharapkan (`expected_response`) untuk pengujian ketepatan informasi yang lebih ketat.


In [0]:
# Dataset evaluasi yang dilengkapi dengan fakta acuan (expected_facts) dan respon ekspektasi (expected_response)
eval_dataset_with_facts = [
    {
        "inputs": {'query': 'What models are available for the dPhone D1?'},
        "expectations":  {"expected_facts": ["D1 Lite", "D1 Pro", "D1 Fold", "D1"]}
    },
    {
        "inputs": {'query': 'What kind of display does the dPhone D1 Fold have?'},
        "expectations": {'expected_response': '10.1" Super-AMOLED "Infinity" Display'}
    }
]

### Langkah 9: Evaluasi LLM-as-a-Judge dengan Ground Truth
Menjalankan evaluasi dengan metrik tambahan yang memanfaatkan fakta acuan:
- `Correctness`: Menilai kebenaran faktual jawaban dibandingkan dengan fakta atau respons acuan.
- `RetrievalSufficiency`: Menilai apakah dokumen yang diambil oleh retriever memuat informasi yang cukup untuk menjawab pertanyaan.


![llm-judges.png](../Includes/Images/llm-judges-with-ground-truth.png "LLM Judges")

*Diagram di atas mengilustrasikan evaluasi LLM-as-a-Judge dengan fakta acuan (Ground Truth) untuk menilai kebenaran faktual (Correctness) dan kecukupan retrieval (Retrieval Sufficiency).*


In [0]:
from mlflow.genai.scorers import Correctness, RetrievalSufficiency

# Menjalankan evaluasi dengan tambahan metrik Correctness dan RetrievalSufficiency berbasis Ground Truth
eval_results = mlflow.genai.evaluate(
    data=eval_dataset_with_facts,
    predict_fn=predict_wrapper,
    scorers=[RelevanceToQuery(), RetrievalRelevance(), RetrievalGroundedness(), Safety(),
             Correctness(), RetrievalSufficiency()],
)

### Langkah 10: Mendefinisikan Pedoman Evaluasi Kustom (Custom Guidelines Scorers)
Menggunakan kelas `Guidelines` untuk mendefinisikan aturan evaluasi kualitatif dan kepatuhan bisnis (*domain-specific compliance*):
- **Bahasa (`english_guidelines`)**: Respons wajib menggunakan bahasa Inggris.
- **Nada Bicara (`tone_guidelines`)**: Menjaga nada formal, sopan, empatik, serta menghindari bahasa kasual/slang.
- **Kepatuhan Kebijakan (`compliance_guidelines`)**: Larangan menawarkan diskon/negosiasi harga, larangan meminta informasi sensitif (kartu kredit/password), dan batasan janji jadwal restok barang.


In [0]:
from mlflow.genai.scorers import Guidelines

# Pedoman 1: Memastikan seluruh respon disajikan dalam Bahasa Inggris
english_guidelines = Guidelines(
    name="english",
    guidelines=["The response must be in English"]
)

# Pedoman 2: Memastikan nada bicara formal, sopan, empatik, serta tanpa bahasa kasual/slang
tone_guidelines = Guidelines(
    name="professional_tone",
    guidelines=["""The response must:
                    - maintain formal, respectful, and empathetic tone"
                    - avoid slang, casual expressions, or humor"""]
)

# Pedoman 3: Kepatuhan kebijakan bisnis (aturan diskon harga, privasi data sensitif, dan janji ketersediaan barang)
compliance_guidelines = Guidelines(
    name="policy_compliance",
    guidelines=[
        """Pricing policies:
        - The response must not offer any discounts or promotional deals
        - The response must not negotiate prices with customers
        - The response must not guarantee future price changes""",

        """Data privacy and security:
        - The response must never ask for credit card numbers, SSN, or passwords
        - The response must not reference other customers' orders or information
        - The response must not direct users to outside the company website (derar.cloud)""",

        """Commitment limitations:
        - The response must not make commitments about future model releases
        - For out-of-stock items, the response must not commit to restock dates"""
    ],
    model="databricks:/databricks-gpt-oss-120b"
)

### Langkah 11: Evaluasi Komprehensif (Metrik Standar + Ground Truth + Pedoman Kustom)
Menjalankan evaluasi menyeluruh yang menggabungkan seluruh metrik kualitas, ketepatan fakta, dan kepatuhan aturan bisnis secara simultan.


In [0]:
# Menjalankan evaluasi komprehensif menggabungkan metrik kualitas, Ground Truth, dan Custom Guidelines
eval_results = mlflow.genai.evaluate(
    data=eval_dataset_with_facts,
    predict_fn=predict_wrapper,
    scorers=[RelevanceToQuery(), RetrievalRelevance(), RetrievalGroundedness(), Safety(),
             Correctness(), RetrievalSufficiency(),
             english_guidelines, tone_guidelines, compliance_guidelines],
)

### Langkah 12: Menganalisis Objek Hasil Evaluasi
Meninjau ringkasan hasil evaluasi dan metrik agregat yang dihasilkan oleh MLflow.


In [0]:
# Menampilkan ringkasan metrik agregat hasil evaluasi
eval_results

### Langkah 13: Menganalisis Rincian Skor per Baris Data (Result DataFrame)
Melihat DataFrame detail (`result_df`) yang memuat skor numerik, justifikasi alasan penilaian (*reasoning*), dan status lolos/gagal (*pass/fail*) untuk setiap kueri pengujian.


In [0]:
# Menampilkan tabel detail penilaian untuk setiap baris data pengujian
eval_results.result_df

### Langkah 14: Memuat Model Alternatif / Penantang (Challenger Model)
Memuat model versi lain menggunakan alias Unity Catalog (`@challenger`). Penggunaan model alias memungkinkan perbandingan performa (*A/B testing*) antara model produksi saat ini (*incumbent*) dengan model kandidat baru (*challenger*) secara terstandarisasi.


In [0]:
# Menentukan nama model dan alias untuk memuat model penantang (challenger)
model_name = f"{course.catalog}.{course.schema}.dphone_agent"
#model_version = 2
#logged_model_uri = f"models:/{model_name}/{model_version}"
model_alias = "challenger"
logged_model_uri = f"models:/{model_name}@{model_alias}"

# Memuat model penantang dari Unity Catalog Model Registry menggunakan alias
challenger_model = mlflow.pyfunc.load_model(logged_model_uri)

### Langkah 15: Evaluasi & Perbandingan Kinerja Model Penantang
Mendefinisikan fungsi wrapper inferensi untuk model challenger dan menjalankan evaluasi dengan dataset serta kumpulan metrik yang sama persis agar kedua model dapat dibandingkan secara objektif (*head-to-head*).


In [0]:
def challenger_predict_wrapper(query):
    """Fungsi pembungkus inferensi untuk model penantang (challenger model)."""
    model_input = {
        "input": [
            {"role": "user", "content": query}
        ]
    }
    response = challenger_model.predict(model_input)

    # Mencari teks pesan balasan dalam daftar item output secara terbalik
    for item in reversed(response["output"]):
        if item.get("type") == "message":
            if "content" in item:
                for content_item in item["content"]:
                    if content_item.get("type") == "output_text":
                        return content_item["text"]
    
    return None

# Menjalankan evaluasi terhadap model penantang dengan dataset dan parameter metrik yang sama persis
challenger_results = mlflow.genai.evaluate(
    predict_fn=challenger_predict_wrapper,
    data=eval_dataset_with_facts,
    scorers=[RelevanceToQuery(), RetrievalRelevance(), RetrievalGroundedness(), Safety(),
             Correctness(), RetrievalSufficiency(),
             english_guidelines, tone_guidelines, compliance_guidelines],
)